In [ ]:
from srai.loaders import HFLoader
import json
import os
import geopandas as gpd
from shapely import LineString, MultiPoint
import pandas as pd

In [ ]:
with open("..\..\srai\loaders\secrets.json") as secrets:
    secrets = json.load(secrets)

In [ ]:
token = secrets["HF_access_token"]

In [ ]:
hf_loader = HFLoader(hf_token=token)

In [ ]:
data = hf_loader.load(dataset_name="nyc_bike", name="nyc_bike_2014")

In [ ]:
type(data)


def make_geodataframe_previous(dataframe: pd.DataFrame) -> gpd.GeoDataFrame:
    start_station_geometry = gpd.points_from_xy(
        x=dataframe["start station longitude"], y=dataframe["start station latitude"]
    )
    end_station_geometry = gpd.points_from_xy(
        x=dataframe["end station longitude"], y=dataframe["end station latitude"]
    )
    multi_point_stations_geometries = [
        MultiPoint([start, end]) for start, end in zip(start_station_geometry, end_station_geometry)
    ]
    gdf = gpd.GeoDataFrame(
        dataframe.drop(
            [
                "start station latitude",
                "start station longitude",
                "end station latitude",
                "end station longitude",
            ],
            axis=1,
        ),
        geometry=multi_point_stations_geometries,
        crs="EPSG:4326",
    )

    return gdf

In [ ]:
gdf = make_geodataframe_previous(data)

In [ ]:
data.head()

In [ ]:
data = hf_loader.load(dataset_name="geolife")

In [ ]:
data["geometry"] = data["arrays_geometry"].map(LineString)
geolife = gpd.GeoDataFrame(
    data.drop(["arrays_geometry"], axis=1),
    geometry=data["geometry"],
    crs="EPSG:4326",
)

In [ ]:
geolife.head()